In [1]:
import torch
import os
import gc

# custom dataloader
from src.dataloader import CSIRODataModule
# models
from src.ResNet_RB import ResNet_RB
from src.ResNet_RRDB import ResNet_RRDB
# model wrapper (to tile the bigger input image)
from src.model_wrapper import ModelWrapper
# training functions
from src.training_functions import train_predictor

from src.feature_extractor_wrapper import FeatureExtractorWrapper
from src.hybrid_model import HybridModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
data_dir = os.path.join(os.getcwd(), "data")
print(data_dir)
dataloader = CSIRODataModule(data_dir=data_dir, image_resize=(448, 896))
dataloader.setup()

c:\Users\szige\Google Drive\02_study\0_BME_alkmat\semester_1\ai_in_ds\kaggle_csiro\data


In [3]:
NUM_EPOCHS = 3
LR = 1e-4
tile_size = 224

fe_model = FeatureExtractorWrapper(
    'EfficientNet-B0', 
    target_dim=128, 
    dropout_rate=0.2, 
    trainable=False
)
hybrid_model = HybridModel(
    dino_model_name='dinov2_vits14',
    fe_model=fe_model, 
    tilesize=tile_size,
    linear_layers=[512, 256],
    num_outputs=5
)

# training
hybrid_model.to(device)
hybrid_model = train_predictor(
    dataloader=dataloader,
    batch_size=2,
    wrapped_model=hybrid_model,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    weight_decay=1e-4,
    use_wandb=False
)

c:\Users\szige\Google Drive\02_study\0_BME_alkmat\semester_1\ai_in_ds\kaggle_csiro\myenv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\szige\Google Drive\02_study\0_BME_alkmat\semester_1\ai_in_ds\kaggle_csiro\myenv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Using cache found in C:\Users\szige/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\szige/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warn

Using device: cuda
Training Hybrid_dinov2_vits14_EfficientNet-B0_and_lin128_224x224_linFlatten(start_dim=1, end_dim=-1)-Linear(in_features=512, out_features=512, bias=True)-ReLU()-Dropout(p=0.5, inplace=False)-Linear(in_features=512, out_features=256, bias=True)-ReLU()-Dropout(p=0.3, inplace=False)-Linear(in_features=256, out_features=5, bias=True)...


Epoch 1/3: 100%|██████████| 161/161 [03:03<00:00,  1.14s/it, R2_loss=1.1958, R2_loss_log1p=0.8545, mse=3044.67]


Validation after epoch 1: R2=-1.67, R_loss=2.67, R_loss_log1p=0.66, mse=962.02
✓ New best model found! R² loss: 2.6696


Epoch 2/3: 100%|██████████| 161/161 [03:01<00:00,  1.13s/it, R2_loss=0.3247, R2_loss_log1p=0.8044, mse=124.50] 


Validation after epoch 2: R2=0.33, R_loss=0.67, R_loss_log1p=0.52, mse=369.31
✓ New best model found! R² loss: 0.6745


Epoch 3/3: 100%|██████████| 161/161 [03:05<00:00,  1.15s/it, R2_loss=0.3634, R2_loss_log1p=0.4688, mse=105.21] 


Validation after epoch 3: R2=0.30, R_loss=0.70, R_loss_log1p=0.53, mse=373.33
No improvement for 1 epoch(s)

Loaded best model with validation R² loss: 0.6745


In [5]:
# clear GPU memory, use it when training stopped
gc.collect()
torch.cuda.empty_cache()